# FIFA World Cup Poisson Model v2 — Feature Engineering & Analysis

This notebook explores an expanded feature set for predicting goals scored per team in World Cup matches.

**Baseline model**: Single `win_exp` feature → PoissonRegressor (mean Poisson deviance: ~1.1585)

**Goal**: Systematically add features and measure their incremental impact via leave-one-tournament-out CV.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_poisson_deviance

from model.cv import leave_one_tournament_out_cv
from model.preprocessing import process_tournament_history
from model.transformers import WinExpTransformer, FullFeatureTransformer
from model.pipelines import build_baseline_pipeline, build_full_pipeline

In [ ]:
df_base = pd.read_json('./data/dataset.json')
print(f'Dataset: {df_base.shape[0]} rows, {df_base.tournament_id.nunique()} tournaments')
print(f'Tournaments: {sorted(df_base.tournament_id.unique())}')
print(f'\nScore distribution:')
df_base['score'].describe()

## 0. Baseline Model

The baseline uses a single feature: `win_exp = 1 / (1 + (rank / opp_rank) ** shape)`

Using the best hyperparameters from the original training (shape=1.4223, alpha=0.000253).

In [ ]:
X_base = df_base[['rank', 'opp_rank']].values
y = df_base['score'].values
t_ids = df_base['tournament_id']

baseline = build_baseline_pipeline(shape=1.4223, alpha=0.000253, max_iter=711)
baseline_score = leave_one_tournament_out_cv(baseline, X_base, y, t_ids)
print(f'Baseline mean Poisson deviance: {baseline_score:.6f}')

# Store results for comparison table
results = [{'features': 'Baseline (win_exp only)', 'score': baseline_score, 'delta': 0.0}]

---
## 1. Current Tournament Rank (cur_win_exp)

Rather than using static FIFA rankings, we evolve ranks during the tournament based on match results using an Elo-style update. This captures **momentum** — a team beating stronger opponents sees their current rank improve.

The EDA notebook (v1) showed that current-rank-based win expectation achieves AIC=2391.64 vs 2419.24 for base-rank — a 27.6 point improvement.

In [ ]:
# Run preprocessing with default parameters
df_proc = process_tournament_history(df_base, shape=1.5, k_mul=5, k_off_mul=5, k_def_mul=5, goal_cap=4.0)

# Show how ranks evolve for Argentina WC-2022 (lost first game, then won 6 straight)
arg = df_proc[(df_proc.tournament_id == 'WC-2022') & (df_proc.team == 'Argentina')]
print('Argentina WC-2022 — Rank Evolution:')
arg[['team', 'opp_team', 'stage_name', 'rank', 'cur_rank', 'score', 'win']].reset_index(drop=True)

In [ ]:
# Visualize: base rank vs current rank at kickoff
fig = px.scatter(
    df_proc, x='rank', y='cur_rank', color='tournament_id',
    hover_data=['team', 'opp_team', 'score'],
    title='Base Rank vs Current (Evolved) Rank at Kickoff',
    template='plotly_dark', opacity=0.6,
    labels={'rank': 'Base FIFA Rank', 'cur_rank': 'Current Tournament Rank'}
)
fig.add_trace(go.Scatter(x=[0, 110], y=[0, 110], mode='lines', 
                          line=dict(dash='dash', color='gray'), 
                          name='Identity line', showlegend=False))
fig.update_layout(width=800, height=600)
fig.show()

In [ ]:
# CV: baseline win_exp + cur_win_exp
FEATURE_COLUMNS = [
    'rank', 'opp_rank', 'cur_rank', 'opp_cur_rank',
    'rank_shift', 'opp_rank_shift',
    'off_rank', 'opp_off_rank', 'def_rank', 'opp_def_rank',
    'host', 'is_strong_confed', 'opp_is_strong_confed',
    'rest_diff', 'stage_weight',
]
X_full = df_proc[FEATURE_COLUMNS].values
y_full = df_proc['score'].values
t_ids_full = df_proc['tournament_id'].reset_index(drop=True)

# Just win_exp + cur_win_exp (disable other features)
pipe_cur = build_full_pipeline(
    shape=1.0, alpha=0.001, max_iter=500,
    include_off_def=False, include_confed=False,
    include_rest=False, include_stage_weight=False
)
score_cur = leave_one_tournament_out_cv(pipe_cur, X_full, y_full, t_ids_full)
print(f'+ cur_win_exp: {score_cur:.6f} (delta: {baseline_score - score_cur:+.6f})')
results.append({'features': '+ cur_win_exp', 'score': score_cur, 'delta': baseline_score - score_cur})

---
## 2. Rank Shift (Team Form)

`rank_shift = cur_rank - rank` measures how much a team's rank has changed during the tournament so far.

- **Negative shift** → rank improved → team is performing above expectations (hot form)
- **Positive shift** → rank worsened → team is underperforming (cold form)

The hypothesis is that a team on a hot streak may score more goals, and facing a hot-streak opponent may result in fewer goals.

In [ ]:
# Distribution of rank shifts
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Team Rank Shift Distribution', 'Rank Shift vs Goals Scored'
))

fig.add_trace(go.Histogram(x=df_proc['rank_shift'], nbinsx=40, 
              marker_color='cyan', opacity=0.7, name='rank_shift'), row=1, col=1)

# Bucket rank_shift and show avg goals
df_proc['rs_bucket'] = pd.cut(df_proc['rank_shift'], bins=np.arange(-30, 35, 5), right=False)
rs_goals = df_proc.groupby('rs_bucket', observed=False)['score'].agg(['mean', 'count']).reset_index()
rs_goals['mid'] = rs_goals['rs_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)
rs_goals = rs_goals.dropna()

fig.add_trace(go.Bar(x=rs_goals['mid'], y=rs_goals['mean'], 
              marker_color='magenta', opacity=0.7, name='avg goals',
              text=rs_goals['count'].astype(str), textposition='outside'), row=1, col=2)

fig.update_layout(template='plotly_dark', height=400, width=1200, showlegend=False)
fig.update_xaxes(title_text='Rank Shift (cur_rank - rank)', row=1, col=1)
fig.update_xaxes(title_text='Rank Shift Bucket', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Avg Goals Scored', row=1, col=2)
fig.show()

In [ ]:
# CV: + rank_shift, opp_rank_shift (already included via FullFeatureTransformer)
# The FullFeatureTransformer always includes rank_shift and opp_rank_shift
# So this test is the same as feature1 (they were already included)
# Let's verify by checking the score - it should be same as feature1 since
# the transformer always passes through rank_shift
print(f'Note: rank_shift and opp_rank_shift are always included with cur_win_exp.')
print(f'Score with win_exp + cur_win_exp + rank_shifts: {score_cur:.6f}')
print(f'These features are bundled — the rank_shift is derived from the same cur_rank computation.')

---
## 3. Offensive Rank

Offensive rank starts at the team's base FIFA rank and evolves based on goals scored, weighted by opponent quality.

**Formula**: Similar Elo-style update, but actual performance = goals_scored / goal_cap.
- Scoring 4+ goals against a top-10 team: massive improvement
- Scoring 0 against a bottom-ranked team: significant worsening

We compute `off_win_exp = 1/(1+(off_rank / opp_def_rank)^shape)` — the team's attacking ability vs the opponent's defensive ability.

In [ ]:
# Show offensive rank evolution for key teams
teams_to_show = ['France', 'Argentina', 'Germany', 'Brazil']
tournaments_to_show = ['WC-2014', 'WC-2018', 'WC-2022']

fig = make_subplots(rows=1, cols=len(tournaments_to_show), 
                    subplot_titles=tournaments_to_show)

colors = {'France': 'cyan', 'Argentina': 'magenta', 'Germany': 'yellow', 'Brazil': 'lime'}

for col, t_id in enumerate(tournaments_to_show, 1):
    for team in teams_to_show:
        team_data = df_proc[(df_proc.tournament_id == t_id) & (df_proc.team == team)]
        if len(team_data) > 0:
            fig.add_trace(go.Scatter(
                x=list(range(1, len(team_data)+1)), y=team_data['off_rank'],
                mode='lines+markers', name=team,
                line=dict(color=colors[team]),
                showlegend=(col == 1),
                legendgroup=team,
            ), row=1, col=col)
    fig.update_xaxes(title_text='Match #', row=1, col=col)

fig.update_yaxes(title_text='Offensive Rank', row=1, col=1)
fig.update_layout(template='plotly_dark', height=400, width=1400, 
                  title='Offensive Rank Evolution During Tournament')
fig.show()

In [ ]:
# off_win_exp vs actual goals scored
shape = 1.0
df_proc['off_win_exp'] = 1 / (1 + (df_proc['off_rank'] / df_proc['opp_def_rank']) ** shape)

# Bucket and plot
df_proc['off_we_bucket'] = pd.cut(df_proc['off_win_exp'], bins=np.arange(0, 1.1, 0.1))
off_goals = df_proc.groupby('off_we_bucket', observed=False)['score'].agg(['mean','count']).reset_index()
off_goals['mid'] = off_goals['off_we_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)
off_goals = off_goals.dropna()

fig = px.bar(off_goals, x='mid', y='mean', text='count',
             title='Offensive Win Expectation vs Avg Goals Scored',
             labels={'mid': 'off_win_exp (bucketed)', 'mean': 'Avg Goals'},
             template='plotly_dark')
fig.update_traces(textposition='outside')
fig.update_layout(width=800, height=400)
fig.show()

In [ ]:
# CV: + off/def win_exp
pipe_offdef = build_full_pipeline(
    shape=1.0, alpha=0.001, max_iter=500,
    include_off_def=True, include_confed=False,
    include_rest=False, include_stage_weight=False
)
score_offdef = leave_one_tournament_out_cv(pipe_offdef, X_full, y_full, t_ids_full)
print(f'+ off_win_exp + def_win_exp: {score_offdef:.6f} (delta: {baseline_score - score_offdef:+.6f})')
results.append({'features': '+ off/def win_exp', 'score': score_offdef, 'delta': baseline_score - score_offdef})

---
## 4. Defensive Rank

Same structure as offensive rank, but based on goals **conceded**. A clean sheet against a strong opponent improves (lowers) the defensive rank more than against a weak opponent.

We compute `def_win_exp = 1/(1+(def_rank / opp_off_rank)^shape)` — the team's defensive ability vs the opponent's attacking ability.

In [ ]:
# Defensive rank evolution for key teams
fig = make_subplots(rows=1, cols=len(tournaments_to_show), 
                    subplot_titles=tournaments_to_show)

for col, t_id in enumerate(tournaments_to_show, 1):
    for team in teams_to_show:
        team_data = df_proc[(df_proc.tournament_id == t_id) & (df_proc.team == team)]
        if len(team_data) > 0:
            fig.add_trace(go.Scatter(
                x=list(range(1, len(team_data)+1)), y=team_data['def_rank'],
                mode='lines+markers', name=team,
                line=dict(color=colors[team]),
                showlegend=(col == 1),
                legendgroup=team,
            ), row=1, col=col)
    fig.update_xaxes(title_text='Match #', row=1, col=col)

fig.update_yaxes(title_text='Defensive Rank', row=1, col=1)
fig.update_layout(template='plotly_dark', height=400, width=1400,
                  title='Defensive Rank Evolution During Tournament')
fig.show()

In [ ]:
# def_win_exp vs actual goals scored (should be negative relationship)
df_proc['def_win_exp'] = 1 / (1 + (df_proc['def_rank'] / df_proc['opp_off_rank']) ** shape)

df_proc['def_we_bucket'] = pd.cut(df_proc['def_win_exp'], bins=np.arange(0, 1.1, 0.1))
def_goals = df_proc.groupby('def_we_bucket', observed=False)['score'].agg(['mean','count']).reset_index()
def_goals['mid'] = def_goals['def_we_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)
def_goals = def_goals.dropna()

fig = px.bar(def_goals, x='mid', y='mean', text='count',
             title='Opponent Defensive Win Exp vs Avg Goals Scored (opponent has strong defense → fewer goals)',
             labels={'mid': 'def_win_exp (bucketed)', 'mean': 'Avg Goals'},
             template='plotly_dark')
fig.update_traces(textposition='outside')
fig.update_layout(width=800, height=400)
fig.show()

---
## 5. Host Nation Advantage

Host nations have a well-documented home advantage. Let's quantify it.

In [ ]:
# Host nation stats
host_stats = df_proc.groupby('host').agg(
    avg_goals=('score', 'mean'),
    win_rate=('win', 'mean'),
    n=('score', 'count')
).reset_index()
host_stats['host'] = host_stats['host'].map({0: 'Non-Host', 1: 'Host'})
print('Host Nation Advantage:')
print(host_stats.to_string(index=False))
print(f'\nHost avg goals: {df_proc[df_proc.host==1]["score"].mean():.2f}')
print(f'Non-host avg goals: {df_proc[df_proc.host==0]["score"].mean():.2f}')
print(f'Host win rate: {df_proc[df_proc.host==1]["win"].mean():.1%}')
print(f'Sample size: {df_proc["host"].sum()} host rows out of {len(df_proc)} total')

In [ ]:
# CV: + host (without off/def to isolate host effect)
# Note: host is always included in FullFeatureTransformer, so to test
# incremental impact we compare with and without other features
print('Host is always included in the full feature set.')
print('Its effect is captured within the overall model improvement.')
print(f'\nWith sample size of only {df_proc["host"].sum()} host matches,') 
print('the regularization (alpha) will appropriately shrink the coefficient.')

---
## 6. Confederation Features

Do teams from certain confederations score more/fewer goals? The hypothesis is that UEFA and CONMEBOL teams tend to score more, while CAF and AFC teams tend to score fewer.

We compare two approaches:
- **Option A**: Binary `is_strong_confed` (UEFA/CONMEBOL vs rest) — 2 features
- **Option B**: No confederation features at all

In [ ]:
# Confederation scoring patterns
confed_stats = df_proc.groupby('confederation').agg(
    avg_goals=('score', 'mean'),
    avg_rank=('rank', 'mean'),
    win_rate=('win', 'mean'),
    n=('score', 'count')
).sort_values('avg_goals', ascending=False)
print('Goals by Confederation:')
print(confed_stats.to_string())

print(f'\nStrong confeds (UEFA+CONMEBOL) avg goals: {df_proc[df_proc.is_strong_confed==1]["score"].mean():.3f}')
print(f'Other confeds avg goals: {df_proc[df_proc.is_strong_confed==0]["score"].mean():.3f}')

In [ ]:
# Confederation matchup heatmap
matchup = df_proc.groupby(['confederation', 'opp_confederation'])['score'].mean().reset_index()
matchup_pivot = matchup.pivot(index='confederation', columns='opp_confederation', values='score')

fig = px.imshow(
    matchup_pivot.values,
    x=matchup_pivot.columns.tolist(),
    y=matchup_pivot.index.tolist(),
    text_auto='.2f',
    color_continuous_scale='RdYlGn',
    title='Avg Goals Scored: Team Confed (rows) vs Opponent Confed (cols)',
    template='plotly_dark'
)
fig.update_layout(width=700, height=500, coloraxis_showscale=False)
fig.show()

In [ ]:
# CV: compare with and without confederation features
pipe_with_confed = build_full_pipeline(
    shape=1.0, alpha=0.001, max_iter=500,
    include_off_def=True, include_confed=True,
    include_rest=False, include_stage_weight=False
)
score_confed = leave_one_tournament_out_cv(pipe_with_confed, X_full, y_full, t_ids_full)

pipe_no_confed = build_full_pipeline(
    shape=1.0, alpha=0.001, max_iter=500,
    include_off_def=True, include_confed=False,
    include_rest=False, include_stage_weight=False
)
score_no_confed = leave_one_tournament_out_cv(pipe_no_confed, X_full, y_full, t_ids_full)

print(f'With confederation features: {score_confed:.6f}')
print(f'Without confederation:       {score_no_confed:.6f}')
print(f'Confederation impact:        {score_no_confed - score_confed:+.6f}')
results.append({'features': '+ off/def + confed', 'score': score_confed, 'delta': baseline_score - score_confed})

---
## 7. Rest Days Difference

`rest_diff = lst_match_days_ago - opp_lst_match_days_ago`

A team with more rest days may perform better. The effect is most visible in knockout rounds where scheduling can create rest imbalances.

In [ ]:
# Rest days analysis
rest_stats = df_proc.groupby('rest_diff').agg(
    avg_goals=('score', 'mean'),
    win_rate=('win', 'mean'),
    n=('score', 'count')
).reset_index()
print('Rest Difference Impact:')
print(rest_stats.to_string(index=False))

fig = px.bar(rest_stats[rest_stats.n >= 5], x='rest_diff', y='avg_goals', text='n',
             title='Rest Days Advantage vs Avg Goals (min 5 samples)',
             labels={'rest_diff': 'Rest Advantage (days)', 'avg_goals': 'Avg Goals'},
             template='plotly_dark')
fig.update_traces(textposition='outside')
fig.update_layout(width=800, height=400)
fig.show()

In [ ]:
# CV: + rest_diff
pipe_rest = build_full_pipeline(
    shape=1.0, alpha=0.001, max_iter=500,
    include_off_def=True, include_confed=True,
    include_rest=True, include_stage_weight=False
)
score_rest = leave_one_tournament_out_cv(pipe_rest, X_full, y_full, t_ids_full)
print(f'+ rest_diff: {score_rest:.6f} (delta: {baseline_score - score_rest:+.6f})')
results.append({'features': '+ rest_diff', 'score': score_rest, 'delta': baseline_score - score_rest})

---
## 8. Competition Stage Weight

Rather than a raw match count or a binary flag, we encode the stage of the competition as an ordinal:

| Stage | Weight |
|-------|--------|
| Group stage | 0 |
| Round of 16, Quarter-finals, Third-place match | 1 |
| Semi-finals, Final | 2 |

The reasoning is that semi-finals and finals are the highest-stakes games, where teams tend to be the most evenly matched and most defensively cautious. The third-place match is a consolation game — data shows it has the *highest* average goals (1.79) of any stage, so it correctly sits in the middle bucket rather than with the high-pressure finals.

In [ ]:
# Goals per stage and stage_weight
stage_stats = df_proc.groupby('stage_name').agg(
    avg_goals=('score', 'mean'),
    n=('score', 'count'),
    stage_weight=('stage_weight', 'first')
).sort_values('stage_weight').reset_index()
print('Goals by Stage:')
print(stage_stats[['stage_name', 'stage_weight', 'avg_goals', 'n']].to_string(index=False))

sw_stats = df_proc.groupby('stage_weight').agg(
    avg_goals=('score', 'mean'),
    win_rate=('win', 'mean'),
    n=('score', 'count')
).reset_index()
sw_stats['label'] = sw_stats['stage_weight'].map({
    0: '0 — Group stage',
    1: '1 — R16 / QF / 3rd-place',
    2: '2 — SF / Final'
})
print('\nGoals by Stage Weight:')
print(sw_stats[['label', 'avg_goals', 'n']].to_string(index=False))

fig = px.bar(
    sw_stats, x='label', y='avg_goals', text='n',
    title='Avg Goals by Competition Stage Weight',
    labels={'label': 'Stage Weight', 'avg_goals': 'Avg Goals'},
    template='plotly_dark', color='avg_goals',
    color_continuous_scale='RdYlGn'
)
fig.update_traces(textposition='outside')
fig.update_layout(width=600, height=400, showlegend=False, coloraxis_showscale=False)
fig.show()

In [ ]:
# CV: + stage_weight (full feature set)
pipe_all = build_full_pipeline(
    shape=1.0, alpha=0.001, max_iter=500,
    include_off_def=True, include_confed=True,
    include_rest=True, include_stage_weight=True
)
score_all = leave_one_tournament_out_cv(pipe_all, X_full, y_full, t_ids_full)
print(f'+ stage_weight: {score_all:.6f} (delta vs baseline: {baseline_score - score_all:+.6f})')
results.append({'features': 'All features (default params)', 'score': score_all, 'delta': baseline_score - score_all})

---
## 9. Incremental Feature Comparison

Summary of all feature experiments with default (untuned) hyperparameters.

In [ ]:
df_results = pd.DataFrame(results)
df_results['delta_pct'] = (df_results['delta'] / baseline_score * 100).round(2)
print('Feature Impact Summary (Leave-One-Tournament-Out CV, default hyperparameters):')
print('=' * 80)
for _, r in df_results.iterrows():
    print(f"  {r['features']:40s}  {r['score']:.6f}  ({r['delta']:+.6f}, {r['delta_pct']:+.2f}%)")
print('=' * 80)

---
## 10. Optuna Hyperparameter Tuning

Now we tune all hyperparameters (preprocessing + transformer + regressor) jointly using Optuna.

**Tunable parameters (8):**
- `preprocess_shape` (0.5-3.0): Elo curve steepness for rank evolution
- `k_mul` (1-15): General rank shift magnitude
- `k_off_mul` (1-15): Offensive rank shift magnitude
- `k_def_mul` (1-15): Defensive rank shift magnitude
- `goal_cap` (2-6): Goal normalization cap
- `feature_shape` (0.1-3.0): Win-exp curve steepness for features
- `alpha` (1e-6 to 10, log): L2 regularization
- `max_iter` (100-1000): Solver iterations

In [ ]:
import optuna

FEATURE_COLS = [
    'rank', 'opp_rank', 'cur_rank', 'opp_cur_rank',
    'rank_shift', 'opp_rank_shift',
    'off_rank', 'opp_off_rank', 'def_rank', 'opp_def_rank',
    'host', 'is_strong_confed', 'opp_is_strong_confed',
    'rest_diff', 'stage_weight',
]

def objective(trial):
    preprocess_shape = trial.suggest_float('preprocess_shape', 0.5, 3.0)
    k_mul = trial.suggest_float('k_mul', 1.0, 15.0)
    k_off_mul = trial.suggest_float('k_off_mul', 1.0, 15.0)
    k_def_mul = trial.suggest_float('k_def_mul', 1.0, 15.0)
    goal_cap = trial.suggest_float('goal_cap', 2.0, 6.0)
    feature_shape = trial.suggest_float('feature_shape', 0.1, 3.0)
    alpha = trial.suggest_float('alpha', 1e-6, 10.0, log=True)
    max_iter = trial.suggest_int('max_iter', 100, 1000)

    df_p = process_tournament_history(
        df_base, shape=preprocess_shape, k_mul=k_mul,
        k_off_mul=k_off_mul, k_def_mul=k_def_mul, goal_cap=goal_cap
    )
    X = df_p[FEATURE_COLS].values
    y = df_p['score'].values
    t_ids = df_p['tournament_id'].reset_index(drop=True)

    pipe = build_full_pipeline(shape=feature_shape, alpha=alpha, max_iter=max_iter)
    return leave_one_tournament_out_cv(pipe, X, y, t_ids)

study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=200, show_progress_bar=True)

print(f'\nBest mean Poisson deviance: {study.best_value:.6f}')
print(f'Baseline:                    {baseline_score:.6f}')
print(f'Improvement:                 {baseline_score - study.best_value:.6f} ({(baseline_score - study.best_value)/baseline_score*100:.2f}%)')
print(f'\nBest parameters:')
for k, v in sorted(study.best_params.items()):
    if isinstance(v, float):
        print(f'  {k:20s} = {v:.6f}')
    else:
        print(f'  {k:20s} = {v}')

In [ ]:
# Per-tournament fold breakdown with best parameters
best = study.best_params
df_best = process_tournament_history(
    df_base, shape=best['preprocess_shape'], k_mul=best['k_mul'],
    k_off_mul=best['k_off_mul'], k_def_mul=best['k_def_mul'], 
    goal_cap=best['goal_cap']
)
X_best = df_best[FEATURE_COLS].values
y_best = df_best['score'].values
t_ids_best = df_best['tournament_id'].reset_index(drop=True)

pipe_best = build_full_pipeline(
    shape=best['feature_shape'], alpha=best['alpha'], max_iter=best['max_iter']
)

# Also run baseline per-fold for comparison
baseline_pipe = build_baseline_pipeline(shape=1.4223, alpha=0.000253, max_iter=711)

print(f'{"Tournament":<12} {"Baseline":>10} {"Expanded":>10} {"Delta":>10} {"n":>5}')
print('-' * 52)
for t_id in sorted(t_ids_best.unique()):
    # Baseline
    tmask = df_base['tournament_id'] != t_id
    vmask = df_base['tournament_id'] == t_id
    baseline_pipe.fit(X_base[tmask], y[tmask])
    yp_b = np.clip(baseline_pipe.predict(X_base[vmask]), 1e-6, None)
    sb = mean_poisson_deviance(y[vmask], yp_b)
    
    # Expanded
    tmask2 = t_ids_best != t_id
    vmask2 = t_ids_best == t_id
    pipe_best.fit(X_best[tmask2], y_best[tmask2])
    yp_e = np.clip(pipe_best.predict(X_best[vmask2]), 1e-6, None)
    se = mean_poisson_deviance(y_best[vmask2], yp_e)
    
    print(f'{t_id:<12} {sb:10.4f} {se:10.4f} {sb-se:+10.4f} {vmask2.sum():5d}')

---
## 11. Feature Importance

Extract the PoissonRegressor coefficients from the trained model to see which features contribute most.

In [ ]:
# Train on all data with best params
pipe_final = build_full_pipeline(
    shape=best['feature_shape'], alpha=best['alpha'], max_iter=best['max_iter']
)
pipe_final.fit(X_best, y_best)

# Feature names match FullFeatureTransformer output order
feature_names = [
    'win_exp', 'cur_win_exp', 'rank_shift', 'opp_rank_shift',
    'off_win_exp', 'def_win_exp', 'host',
    'is_strong_confed', 'opp_is_strong_confed',
    'rest_diff', 'stage_weight'
]
coefs = pipe_final.named_steps['poisson'].coef_
intercept = pipe_final.named_steps['poisson'].intercept_

print(f'Intercept: {intercept:.4f}')
print(f'\n{"Feature":25s} {"Coefficient":>12s} {"Abs":>8s}')
print('-' * 48)
coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefs})
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False)
for _, row in coef_df.iterrows():
    print(f"{row['feature']:25s} {row['coef']:+12.4f} {row['abs_coef']:8.4f}")

# Bar chart of coefficients
fig = px.bar(coef_df, x='feature', y='coef', color='coef',
             color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
             title='Poisson Regressor Coefficients (positive = more goals)',
             template='plotly_dark')
fig.update_layout(width=900, height=400, coloraxis_showscale=False)
fig.show()

---
## 12. Reflection: Additional Features to Consider

After exploring the implemented features, here are additional ones that could further improve the model:

### 12.1 Goal Scoring Momentum
A **running average of goals scored** in the last N matches within the tournament. While the offensive rank captures quality-adjusted scoring, a raw momentum signal ("this team has been putting the ball in the net") adds value independently — it captures hot/cold streaks without the Elo dampening effect.

### 12.2 World Cup Experience
Count of **prior World Cup appearances** for each team in the dataset. Teams with more WC experience may perform better (composure, tactical knowledge) or worse (aging squads). This is straightforward to compute directly from the dataset by counting prior `tournament_id` occurrences per team.

### 12.3 Group Stage Pressure
A feature capturing whether a team **must win** to advance from the group stage, computed from accumulated points during the group stage iteration. Teams in must-win situations may play more aggressively — both scoring and conceding more. This would require extending `process_tournament_history` to track group standings.

### 12.4 Opponent's Raw Defensive Record in Tournament
Rather than the opponent's defensive rank (an Elo-style aggregate), track the **raw goals-conceded-per-match** for the opponent so far in the tournament. This directly measures how leaky the opponent's defense has been without the Elo smoothing effect, potentially capturing when a defense has suddenly deteriorated.

### 12.5 Head-to-Head History
Whether these teams have met before in prior World Cups and what the results were. Some matchups may carry psychological weight (e.g., Germany vs Brazil post 7-1). Signal is likely weak given the small number of repeated matchups across seven tournaments, but worth measuring.

### 12.6 Travel / Climate Features (WC 2026 specific)
For the 2026 World Cup, venue-related features like altitude (Mexico City at 2,240m), temperature, or travel distance between match venues could matter. These are not in the historical dataset but could be added as prediction-time features for WC 2026.

### Priority Ranking
1. **Goal scoring momentum** — captures form independently of quality-adjustment; easy to implement within the existing preprocessing loop
2. **World Cup experience** — simple cross-tournament lookup, novel signal not correlated with rank
3. **Group stage pressure** — requires standings logic but captures tactical behaviour directly
4. **Opponent's raw defensive record** — complements the existing defensive rank feature
5. **Head-to-head / climate** — interesting but likely weak signal with current data